# Streaming and async

Read the [guide](README.md). Run cells in order with the notebooks and apps groups. Offline replay uses real SDK and local protocol paths; it does not measure live model quality.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_stream_events

Predict the outcome, then run the [script](examples/01_stream_events.py).


In [ ]:
"""Stream events: Nimbus Tier 3 lesson.

Run: uv run python modules/10_streaming/examples/01_stream_events.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

import asyncio

from strands import Agent, tool

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.testing import tool_message


async def demo(cfg):
    @tool
    def trip_stop() -> str:
        """Return a fixed synthetic stop."""
        return "Boston waterfront"

    callback_keys = []
    scripted = tool_message("trip_stop", {})
    scripted["content"].insert(
        0,
        {
            "reasoningContent": {
                "reasoningText": {"text": "Fixture planning summary: inspect the available stop."}
            }
        },
    )
    agent = Agent(
        model=get_model(cfg, responses=[scripted, "Visit the Boston waterfront."]),
        tools=[trip_stop],
        callback_handler=lambda **e: callback_keys.extend(e),
        retry_strategy=None,
    )
    # KEY IDEA: a stream includes lifecycle/tool events as well as displayable text.
    keys = set()
    async for event in agent.stream_async("Suggest a stop", limits={"turns": 3}):
        keys.update(event)
        if "data" in event:
            console.print(event["data"])
    console.print("Iterator event keys:", sorted(keys))
    console.print("Callback keys:", sorted(set(callback_keys)))
    console.print("Reasoning appears only if the provider exposes it; do not log private input.")


def main():
    _, cfg = lesson_settings()
    asyncio.run(demo(cfg))
    complete()


run_lesson(main, "modules/10_streaming/examples/01_stream_events.py")

## 02_concurrency_and_cancel

Predict the outcome, then run the [script](examples/02_concurrency_and_cancel.py).


In [ ]:
"""Concurrency and cancel: Nimbus Tier 3 lesson.

Run: uv run python modules/10_streaming/examples/02_concurrency_and_cancel.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

import asyncio

from strands import Agent, tool

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import run_cli
from common.streaming import planner
from common.testing import tool_message


async def demo(cfg):
    gate = asyncio.Semaphore(2)

    async def plan(city):
        async with gate:
            return str(await planner(cfg).invoke_async("Plan " + city, limits={"turns": 2}))

    print(await asyncio.gather(*(plan(c) for c in ["Boston", "Porto", "Tokyo"])))
    stopped = asyncio.Event()

    @tool
    async def slow_lookup() -> str:
        """Wait locally to make cancellation observable; creates no external work."""
        try:
            await asyncio.sleep(30)
            return "finished"
        finally:
            stopped.set()

    agent = Agent(
        model=get_model(provider="mock", responses=[tool_message("slow_lookup", {})]),
        tools=[slow_lookup],
        callback_handler=None,
        retry_strategy=None,
    )
    # KEY IDEA: wait_for cancels cooperative async work; it cannot undo external side effects.
    try:
        await asyncio.wait_for(agent.invoke_async("Wait"), timeout=0.15)
    except TimeoutError:
        print("Timed out; async tool cleanup:", stopped.is_set())


def main():
    _, cfg = lesson_settings()
    asyncio.run(demo(cfg))
    complete()


run_lesson(main, "modules/10_streaming/examples/02_concurrency_and_cancel.py")

## 03_fastapi_sse

Predict the outcome, then run the [script](examples/03_fastapi_sse.py).


In [ ]:
"""Fastapi sse: Nimbus Tier 3 lesson.

Run: uv run --group apps python modules/10_streaming/examples/03_fastapi_sse.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

import argparse

from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.web_apps import demonstrate, serve


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--serve", action="store_true")
    args, cfg = lesson_settings(parser)
    # KEY IDEA: replay tests the real ASGI routes; --serve starts the local browser app.
    if args.serve:
        serve(cfg)
    else:
        demonstrate(cfg)
        complete()


run_lesson(main, "modules/10_streaming/examples/03_fastapi_sse.py")

## 04_streamlit_chat

Predict the outcome, then run the [script](examples/04_streamlit_chat.py).


In [ ]:
"""Streamlit chat: Nimbus Tier 3 lesson.

Run: uv run --group apps python modules/10_streaming/examples/04_streamlit_chat.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

import argparse
from contextlib import aclosing

from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.streaming import planner, sync_chunks


async def text_chunks(agent, prompt):
    # A finite fixture must be renewed for each UI turn; retain this session's history.
    from common.testing import ScriptedModel, text_message

    if isinstance(agent.model, ScriptedModel) and agent.model.index == len(agent.model.responses):
        agent.model.responses.append(text_message("Explore Boston on foot; confirm your dates."))
    async with aclosing(agent.stream_async(prompt, limits={"turns": 3})) as events:
        async for event in events:
            if "data" in event:
                yield event["data"]


def render(cfg):
    import streamlit as st

    st.title("Nimbus trip planner")
    if "nimbus_agent" not in st.session_state:
        st.session_state.nimbus_agent = planner(cfg)
        st.session_state.nimbus_chat = []
    for role, message in st.session_state.nimbus_chat:
        with st.chat_message(role):
            st.write(message)
    prompt = st.chat_input("Plan a short trip")
    if prompt:
        st.session_state.nimbus_chat.append(("user", prompt))
        with st.chat_message("user"):
            st.write(prompt)
        with st.chat_message("assistant"):
            text = st.write_stream(sync_chunks(text_chunks(st.session_state.nimbus_agent, prompt)))
        st.session_state.nimbus_chat.append(("assistant", text))
    if st.button("Clear this session"):
        del st.session_state.nimbus_agent
        st.session_state.nimbus_chat = []
        st.rerun()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--ui", action="store_true")
    args, cfg = lesson_settings(parser)
    if args.ui:
        render(cfg)
    else:
        import asyncio

        async def replay():
            return [
                t
                async for t in text_chunks(
                    planner(cfg), "Suggest a Boston trip in under 100 words."
                )
            ]

        print(asyncio.run(replay()))
        complete()


run_lesson(main, "modules/10_streaming/examples/04_streamlit_chat.py")

## streaming_trip_planner

Predict the outcome, then run the [script](use_case/streaming_trip_planner.py).


In [ ]:
"""Eaming trip planner: Nimbus Tier 3 lesson.

Run: uv run python modules/10_streaming/use_case/streaming_trip_planner.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

import argparse
import asyncio

from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.streaming import planner, public_stream


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--serve", action="store_true")
    args, cfg = lesson_settings(parser)
    if args.serve:
        from common.web_apps import serve

        serve(cfg)
    else:

        async def replay():
            async for frame in public_stream(planner(cfg), "Plan a Boston weekend"):
                print(frame, end="")

        asyncio.run(replay())
        complete()


run_lesson(main, "modules/10_streaming/use_case/streaming_trip_planner.py")

## Practice

Complete the [exercises](README.md#exercises) and [quiz](quiz.md), then run the module tests.
